# Python bridge for 02b

## A short code refresher

Use this optional notebook alongside 02b if you would like a closer look at the Python mechanics behind bootstrap resampling. It takes about 15–20 minutes and continues the small two-group example from the 02a Python bridge.

The 02b lecture develops the statistical reasoning. This companion focuses on five code patterns:

1. draw a sample with replacement;
2. calculate one bootstrap replicate;
3. repeat the draw and store many replicates;
4. extract percentile endpoints; and
5. inspect the object returned by SciPy.

It also shows how `np.concatenate` combines pieces that were resampled separately.

## Create two recorded groups

Each array contains seven recorded values. The statistic function accepts two arrays and returns one number: the median of group A minus the median of group B.

In [ ]:
import numpy as np
from scipy.stats import bootstrap

In [ ]:
group_a = np.array([8, 10, 11, 14, 16, 19, 23])
group_b = np.array([5, 7, 9, 10, 12, 13, 15])


def median_difference(x, y, axis=-1):
    return np.median(x, axis=axis) - np.median(y, axis=axis)


point_estimate = median_difference(group_a, group_b)
point_estimate

Calling the function normally uses `axis=-1`, the last axis of each input. Keeping `axis` as a parameter also lets SciPy evaluate batches of resamples efficiently.

## Draw with replacement

`rng.choice` draws values from an array. `size=len(group_a)` requests the same number of values as the recorded group, and `replace=True` allows a recorded value to appear more than once or not at all.

In [ ]:
rng = np.random.default_rng(7130)

resampled_a = rng.choice(group_a, size=len(group_a), replace=True)

print("recorded A:", group_a)
print("resampled A:", resampled_a)

The seed initializes a reproducible sequence. Reusing `rng` advances that sequence, so the next call produces another draw.

## Calculate one bootstrap replicate

A two-group replicate needs one resampled version of each group. The statistic calculated from those two resampled arrays is one bootstrap replicate.

In [ ]:
resampled_b = rng.choice(group_b, size=len(group_b), replace=True)
one_replicate = median_difference(resampled_a, resampled_b)

print("point estimate:", point_estimate)
print("one bootstrap replicate:", one_replicate)

The point estimate comes from the recorded arrays. The replicate comes from one pair of resampled arrays.

## Store many replicates

`np.empty(N_REPLICATES)` allocates one output position for every planned replicate. The loop creates one resampled pair at a time and stores its statistic at `replicates[index]`.

In [ ]:
N_REPLICATES = 1_000
replicate_rng = np.random.default_rng(7130)
replicates = np.empty(N_REPLICATES)

for index in range(N_REPLICATES):
    sampled_a = replicate_rng.choice(group_a, size=len(group_a), replace=True)
    sampled_b = replicate_rng.choice(group_b, size=len(group_b), replace=True)
    replicates[index] = median_difference(sampled_a, sampled_b)

print("replicate shape:", replicates.shape)
print("first five replicates:", replicates[:5])

`range(N_REPLICATES)` supplies the positions from `0` through `N_REPLICATES - 1`. Every position in the initially unfilled array receives one calculated value.

## Extract percentile endpoints

`np.quantile` finds requested positions in the ordered replicate values. Passing `[0.025, 0.975]` returns the lower and upper endpoints of the central 95 percent.

In [ ]:
manual_interval = np.quantile(replicates, [0.025, 0.975])

print("lower endpoint:", manual_interval[0])
print("upper endpoint:", manual_interval[1])

The returned array has two values in the same order as the requested quantiles. Index `0` selects the lower endpoint; index `1` selects the upper endpoint.

## Inspect the SciPy result

SciPy can perform the same repeated workflow. The tuple `(group_a, group_b)` supplies the two samples, and `statistic=median_difference` supplies the function to recalculate.

In [ ]:
result = bootstrap(
    (group_a, group_b),
    statistic=median_difference,
    paired=False,
    n_resamples=1_000,
    confidence_level=0.95,
    method="percentile",
    rng=np.random.default_rng(7130),
)

print("replicate shape:", result.bootstrap_distribution.shape)
print("standard error:", result.standard_error)
print("lower endpoint:", result.confidence_interval.low)
print("upper endpoint:", result.confidence_interval.high)

The returned object groups three kinds of output:

- `result.bootstrap_distribution` is the array of calculated replicates;
- `result.standard_error` is the standard deviation of those replicates; and
- `result.confidence_interval.low` and `.high` are the requested interval endpoints.

The argument `paired=False` tells SciPy to resample the two groups independently. The 02b notebook explains when that structure fits the recorded data.

## Combine separately resampled pieces

A structured bootstrap can resample smaller parts separately. `np.concatenate` joins one-dimensional arrays end to end after those separate draws.

In [ ]:
group_a_region_1 = np.array([8, 11, 16, 23])
group_a_region_2 = np.array([10, 14, 19])
structured_rng = np.random.default_rng(7130)

sampled_region_1 = structured_rng.choice(
    group_a_region_1, size=len(group_a_region_1), replace=True
)
sampled_region_2 = structured_rng.choice(
    group_a_region_2, size=len(group_a_region_2), replace=True
)
combined_a = np.concatenate([sampled_region_1, sampled_region_2])

print("first resampled part:", sampled_region_1)
print("second resampled part:", sampled_region_2)
print("combined group:", combined_a)

The brackets passed to `np.concatenate` contain the arrays to join. Separate sampling preserves each part's original size; concatenation restores one complete group for the statistic calculation.

## Try one small modification

Change `N_REPLICATES` from `1_000` to `2_000`, then rerun the loop and percentile cells. The replicate array should have shape `(2000,)`, and the same two quantile positions should produce the interval endpoints.

## Ready for 02b

You are ready to return to 02b when you can recognize these patterns:

```text
rng.choice(values, size=len(values), replace=True)  # draw one bootstrap sample
replicates = np.empty(n_resamples)                  # allocate the result array
replicates[index] = statistic(sample_a, sample_b)   # store one replicate
np.quantile(replicates, [0.025, 0.975])             # extract percentile endpoints
np.concatenate([part_1, part_2])                    # join separately sampled pieces
result.bootstrap_distribution                       # inspect one SciPy result field
```

The important implementation checks are that each resampled group retains its intended size, the generator is initialized once and then advanced, every loop iteration stores one statistic, and the result field matches the next calculation.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).